# Classical Mechanics: ChatGPT and Numerical Modeling
**MA PH 343 – Classical Mechanics II | University of Alberta**  
**Author:** Rudmila Samuad Eka  
**Date:** December 12, 2025

---

## Project Overview

This project investigates the dynamics of a **hamster + armadillo-in-a-ball system inside a rolling wheel**, using:
- First-principles derivation of the Lagrangian and Euler–Lagrange equations
- Symbolic computation via **SymPy** to obtain explicit equations of motion
- Numerical integration via **SciPy** to simulate the system dynamics
- Critical evaluation of **generative AI (ChatGPT)** outputs for accuracy

The project also documents how much 'directing' was required to get ChatGPT to produce correct results.

---

## Problem Statement

A vertical wheel of radius $R$, mass $M$, and moment of inertia $I$ rolls without slipping on a horizontal table. Inside the wheel:

- A **hamster** of mass $m_H$ runs at constant speed $V$ on a circular track of radius $R - h$
- An **armadillo** rolled in a ball of mass $m_A$ and moment of inertia $I_A$ rolls freely without slipping inside the wheel, also on a circle of radius $R - h$

**Tasks:**
- (a) Derive the Euler–Lagrange equations of motion
- (b) Write a Python program to generate the numerical solution
- (c) Run the program and plot the solution
- Assess how much directing ChatGPT required

---
## Part (a): Deriving the Equations of Motion

### Generalized Coordinates

Two generalized coordinates are chosen:
- $\varphi(t)$: rotation angle of the wheel, measured from the downward vertical
- $\theta(t)$: angle of the vector from wheel centre to armadillo centre, measured from the downward vertical

The configuration space is therefore $\mathcal{Q} = S^1 \times S^1$.

The hamster runs at constant angular speed relative to the wheel:
$$\Omega_H = \frac{V}{R - h}$$
so its absolute angle is $\Omega_H t + \varphi$.

### No-Slip Constraint for the Armadillo Ball

Let $\psi(t)$ be the spin angle of the armadillo ball about its own centre. The no-slip condition between the ball and the inner wheel wall gives:
$$h\dot{\psi} + (R-h)\dot{\theta} = R\dot{\varphi} \implies \dot{\psi} = \frac{R}{h}\dot{\varphi} - \frac{R-h}{h}\dot{\theta}$$

### Kinetic and Potential Energies

$$T_B = \frac{1}{2}(MR^2 + I)\dot{\varphi}^2$$

$$T_H = \frac{1}{2}m_H\left[R^2\dot{\varphi}^2 + (R-h)^2(\dot{\varphi}+\Omega_H)^2 + 2(R-h)R\dot{\varphi}(\dot{\varphi}+\Omega_H)\cos(\Omega_H t + \varphi)\right]$$

$$T_A = \frac{1}{2}m_A\left[R^2\dot{\varphi}^2 + (R-h)^2\dot{\theta}^2 + 2(R-h)R\dot{\varphi}\dot{\theta}\cos\theta\right] + \frac{I_A}{2h^2}\left(R\dot{\varphi} - (R-h)\dot{\theta}\right)^2$$

$$V = -m_H g(R-h)\cos(\Omega_H t + \varphi) - m_A g(R-h)\cos\theta$$

### Time-Independent Lagrangian

To remove explicit time dependence, introduce:
$$q(t) = \varphi(t) + \Omega_H t \implies \dot{\varphi} = \dot{q} - \Omega_H$$

This yields a time-independent Lagrangian $L(q, \theta, \dot{q}, \dot{\theta})$, enabling clean energy conservation analysis.

### Euler–Lagrange Equations

$$\frac{d}{dt}\frac{\partial L}{\partial \dot{q}} - \frac{\partial L}{\partial q} = 0, \qquad \frac{d}{dt}\frac{\partial L}{\partial \dot{\theta}} - \frac{\partial L}{\partial \theta} = 0$$

These yield two coupled nonlinear second-order ODEs for $q(t)$ and $\theta(t)$, solved symbolically below using SymPy.

---
## Part (b): Python Program — Symbolic + Numerical Solution

### Imports

In [ ]:
import numpy as np
import sympy as sp
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

print("Libraries loaded.")

### System Parameters

In [ ]:
# Physical parameters
R   = 0.5                    # wheel radius (m)
h   = 0.10                   # distance from inner wall to ball centre (m)
M   = 2.0                    # wheel mass (kg)
I   = 0.5 * M * R**2         # moment of inertia — solid cylinder
mH  = 0.20                   # hamster mass (kg)
mA  = 0.30                   # armadillo+ball mass (kg)
IA  = 0.4 * mA * h**2        # moment of inertia — approximate solid sphere
V   = 0.5                    # hamster speed along track (m/s)
g   = 9.81                   # gravitational acceleration (m/s²)

OmegaH = V / (R - h)         # hamster angular speed (rad/s)

print(f"Hamster angular speed ΩH = {OmegaH:.4f} rad/s")
print(f"Wheel moment of inertia I = {I:.4f} kg·m²")
print(f"Armadillo moment of inertia IA = {IA:.6f} kg·m²")

### Step 1 — Build the Symbolic Lagrangian

Using the time-independent form $L(q, \theta, \dot{q}, \dot{\theta})$ with substitution $\dot{\varphi} = \dot{q} - \Omega_H$.

In [ ]:
# Symbolic variables
q, th   = sp.symbols('q th', real=True)
qd, thd = sp.symbols('qd thd', real=True)

# Time-independent Lagrangian L(q, theta, qdot, thetadot)
# Note: phi_dot = qd - OmegaH

# Wheel kinetic energy
T_wheel = sp.Rational(1,2) * (M*R**2 + I) * (qd - OmegaH)**2

# Hamster kinetic energy
T_hamster = sp.Rational(1,2) * mH * (
    R**2 * (qd - OmegaH)**2
    + (R - h)**2 * qd**2
    + 2 * (R - h) * R * (qd - OmegaH) * qd * sp.cos(q)
)

# Armadillo kinetic energy (translation + ball rotation)
T_armadillo = sp.Rational(1,2) * mA * (
    R**2 * (qd - OmegaH)**2
    + (R - h)**2 * thd**2
    + 2 * (R - h) * R * (qd - OmegaH) * thd * sp.cos(th)
) + (IA / (2*h**2)) * (R*(qd - OmegaH) - (R - h)*thd)**2

# Potential energy
V_pot = mH * g * (R - h) * sp.cos(q) + mA * g * (R - h) * sp.cos(th)

# Full Lagrangian
L = T_wheel + T_hamster + T_armadillo + V_pot

print("Lagrangian constructed successfully.")

### Step 2 — Derive Euler–Lagrange Equations Symbolically

In [ ]:
qdd, thdd = sp.symbols('qdd thdd', real=True)

def ddt(expr):
    """Total time derivative acting on an expression in (q, th, qd, thd)."""
    return (
        sp.diff(expr, q)   * qd  +
        sp.diff(expr, th)  * thd +
        sp.diff(expr, qd)  * qdd +
        sp.diff(expr, thd) * thdd
    )

# Euler-Lagrange equations: d/dt(dL/dqdot) - dL/dq = 0
EL_q  = sp.simplify(ddt(sp.diff(L, qd))  - sp.diff(L, q))
EL_th = sp.simplify(ddt(sp.diff(L, thd)) - sp.diff(L, th))

# Solve for qddot and thetaddot
print("Solving for accelerations (this may take a moment)...")
sol = sp.solve([EL_q, EL_th], (qdd, thdd), simplify=True)

qdd_expr  = sp.simplify(sol[qdd])
thdd_expr = sp.simplify(sol[thdd])

# Convert to fast numerical functions
qdd_fun  = sp.lambdify((q, th, qd, thd), qdd_expr,  'numpy')
thdd_fun = sp.lambdify((q, th, qd, thd), thdd_expr, 'numpy')

print("Equations of motion derived and compiled successfully.")

### Step 3 — Define ODE System and Integrate Numerically

In [ ]:
def rhs(t, y):
    """Right-hand side of the ODE system: state = [q, theta, qdot, thetadot]"""
    q_val, th_val, qd_val, thd_val = y
    return [
        qd_val,
        thd_val,
        float(qdd_fun(q_val, th_val, qd_val, thd_val)),
        float(thdd_fun(q_val, th_val, qd_val, thd_val))
    ]

# Initial conditions — small displacements from equilibrium (both at bottom)
q0   =  0.05   # rad
th0  = -0.03   # rad
qd0  =  0.0    # rad/s
thd0 =  0.0    # rad/s
y0   = [q0, th0, qd0, thd0]

# Integration parameters
t_span = (0.0, 20.0)
t_eval = np.linspace(*t_span, 2000)

print("Integrating equations of motion...")
sol_ivp = solve_ivp(
    rhs, t_span, y0,
    t_eval=t_eval,
    rtol=1e-8, atol=1e-9
)

print(f"Integration complete. Status: {'Success' if sol_ivp.success else 'Failed'}")
print(f"Time steps computed: {len(sol_ivp.t)}")

---
## Part (c): Results and Visualizations

In [ ]:
t    = sol_ivp.t
q_t  = sol_ivp.y[0]
th_t = sol_ivp.y[1]
qd_t = sol_ivp.y[2]
thd_t= sol_ivp.y[3]

fig, axes = plt.subplots(2, 1, figsize=(11, 8))

# Plot 1: Generalized coordinates
axes[0].plot(t, q_t,  color='#2E86AB', lw=2, label='Hamster angle $q(t)$')
axes[0].plot(t, th_t, color='#E84855', lw=2, label='Armadillo angle $\\theta(t)$')
axes[0].set_xlabel('Time (s)', fontsize=11)
axes[0].set_ylabel('Angle (rad)', fontsize=11)
axes[0].set_title('Hamster and Armadillo in a Rolling Wheel — Generalized Coordinates', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Generalized velocities
axes[1].plot(t, qd_t,  color='#2E86AB', lw=2, label='$\\dot{q}(t)$')
axes[1].plot(t, thd_t, color='#E84855', lw=2, label='$\\dot{\\theta}(t)$')
axes[1].set_xlabel('Time (s)', fontsize=11)
axes[1].set_ylabel('Angular velocity (rad/s)', fontsize=11)
axes[1].set_title('Generalized Velocities', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('hamster_armadillo_solution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")

### Energy Conservation Check

Since the Lagrangian is time-independent, energy should be conserved. We verify this numerically.

In [ ]:
def compute_energy(q_v, th_v, qd_v, thd_v):
    """Compute total mechanical energy at each time step."""
    # Kinetic energy terms
    T_w = 0.5 * (M*R**2 + I) * (qd_v - OmegaH)**2
    T_h = 0.5 * mH * (
        R**2*(qd_v-OmegaH)**2
        + (R-h)**2*qd_v**2
        + 2*(R-h)*R*(qd_v-OmegaH)*qd_v*np.cos(q_v)
    )
    T_a = 0.5 * mA * (
        R**2*(qd_v-OmegaH)**2
        + (R-h)**2*thd_v**2
        + 2*(R-h)*R*(qd_v-OmegaH)*thd_v*np.cos(th_v)
    ) + (IA/(2*h**2))*(R*(qd_v-OmegaH) - (R-h)*thd_v)**2
    # Potential energy
    V_e = mH*g*(R-h)*np.cos(q_v) + mA*g*(R-h)*np.cos(th_v)
    return T_w + T_h + T_a + V_e

E = compute_energy(q_t, th_t, qd_t, thd_t)

plt.figure(figsize=(11, 4))
plt.plot(t, E, color='#3BB273', lw=2)
plt.xlabel('Time (s)', fontsize=11)
plt.ylabel('Total Energy (J)', fontsize=11)
plt.title('Energy Conservation Check', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('energy_conservation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Initial energy:  {E[0]:.6f} J")
print(f"Final energy:    {E[-1]:.6f} J")
print(f"Max deviation:   {np.max(np.abs(E - E[0])):.2e} J")
print(f"Energy conserved: {np.max(np.abs(E - E[0])) < 1e-6}")

---
## Discussion: How Much Directing Did ChatGPT Require?

The project statement predicted that generative AI would likely not solve the problem directly. This was confirmed in practice across three key areas:

### 1. Coordinates and Constraints
When given only the problem statement, ChatGPT chose coordinate conventions different from the textbook and initially **omitted the rotational kinetic energy of the armadillo ball**. It also wrote the no-slip constraint incorrectly. After consulting the book, I explicitly supplied:
$$h\dot{\psi} + (R-h)\dot{\theta} = R\dot{\varphi}$$
and required the AI to use this relation consistently throughout.

### 2. Time-Dependent vs. Time-Independent Lagrangian
ChatGPT initially retained explicit time dependence from the hamster's prescribed motion. I had to direct it to introduce $q = \varphi + \Omega_H t$ to eliminate the time dependence — a step essential for clean energy conservation analysis.

### 3. Symbolic Computation Strategy
When asked for equations of motion, ChatGPT's hand-expanded expressions were long and error-prone. I redirected it to:
- Encode the Lagrangian symbolically in SymPy
- Compute Euler–Lagrange equations by automatic differentiation
- Solve for $\ddot{q}$ and $\ddot{\theta}$ algebraically

### Conclusion
The AI was useful as a **computational and coding assistant**, but only after the physical model, constraints, and coordinate choices were carefully specified and repeatedly checked against the textbook solution. The AI's role was execution, not physical reasoning.

---
*MA PH 343 – Classical Mechanics II · University of Alberta · Mathematical Physics · Science Co-op Program*